<a href="https://colab.research.google.com/github/AnaraHayat/flyrank_assignment1/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [4]:
import os

REPO_URL = "https://github.com/AnaraHayat/flyrank_assignment1.git"
REPO_DIR = "/content/flyrank_assignment1"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

import numpy as np, pandas as pd
from pathlib import Path

DATA_REL = "data/raw/content_refresh_anonymized.csv"
start = Path.cwd()
repo_root = None
for candidate in [start, *start.parents]:
    if (candidate / DATA_REL).exists():
        repo_root = candidate
        break
if repo_root is None:
    raise FileNotFoundError(f"Couldn't find {DATA_REL} above {start}.")
os.chdir(repo_root)

df = pd.read_csv(DATA_REL)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

key_fields = ["days_since_last_update", "impressions_90d", "ctr", "avg_position", "word_count"]
print(df[key_fields].describe(percentiles=[.5, .9, .99]).round(3))
print()

# --- Trap 1: avg_position == 0 does NOT mean 'ranked #0' -- it means 'no position data'. ---
no_pos = (df["avg_position"] == 0).sum()

# --- Trap 2: averaging per-row CTR is not the same as the true (weighted) CTR. ---
simple_mean_ctr = df["ctr"].mean()
weighted_ctr = df["clicks_90d"].sum() / df["impressions_90d"].sum() * 100

Already up to date.
cwd: /content/flyrank_assignment1
       days_since_last_update  impressions_90d        ctr  avg_position  \
count               30000.000        30000.000  30000.000     30000.000   
mean                   46.098         5200.366      0.511        16.342   
std                    42.079        16838.020      3.279        15.217   
min                     1.000            1.000      0.000         0.000   
50%                    20.000          731.000      0.070        10.800   
90%                   104.000        12136.400      0.650        36.800   
99%                   106.000        73505.830      8.330        69.901   
max                   373.000       517715.000    100.000       245.000   

       word_count  
count   22301.000  
mean     3107.760  
std      1452.383  
min         8.000  
50%      2877.000  
90%      5327.000  
99%      7292.000  
max      9546.000  



Heavy tails: impressions_90d median is 731 but the 99th percentile is 73,506 and the max
is 517,715 -- a few giant pages, a long tail of tiny ones. Same shape for ctr (median 0.07%,
max 100%). A plain average over rows like this will be dominated by a handful of outliers --
every test below either weights by a denominator or uses tiers/ranks instead of raw means.

avg_position == 0: 1205 rows -- per the data dictionary this means 'no position data,'
not a perfect position. Left unhandled, these rows land in the 'top_3' position_tier bucket
(0 <= 3) and would fake a chunk of great-looking positions that are actually missing data.
Confirm: 1205 of them are
currently tagged top_3 in the raw data. I exclude avg_position == 0 from every position-based
test below.

Simple mean of per-row ctr: 0.511%
Weighted ctr (total clicks / total impressions): 0.310%
These don't match -- tiny pages with 1-2 impressions and a lucky click post huge, noisy
per-row CTRs (up to 100%) and drag the simple average up. Every CTR comparison below is
weighted (sum clicks / sum impressions), never a mean of per-row ratios

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [6]:
SAMPLE_FLOOR = 50

def verdict_line(name, table_str):
    print(f"=== {name} ===")
    print(table_str)
    print()

# --- Test 1: staleness (behind the refresh flag) ---
# Claim: "stale pages (freshness_tier 91-180 / 181+) are more likely to be declining."
staleness_table = df.groupby("freshness_tier").agg(
    n=("content_id", "size"), decline_rate=("is_declining_label", "mean")
).sort_values("decline_rate")
verdict_line("Test 1: staleness (freshness_tier) vs decline rate", staleness_table.round(3).to_string())

print()

# --- Test 2: volume (behind the quick-win flag) ---
# Claim: "higher-impression pages are more likely to be declining."
volume_table = df.groupby("impression_tier").agg(
    n=("content_id", "size"), decline_rate=("is_declining_label", "mean")
).reindex(["low", "moderate", "good", "excellent"])
verdict_line("Test 2: volume (impression_tier) vs decline rate", volume_table.round(3).to_string())

print()

# --- Test 3: CTR vs position (behind the CTR-fix flag) ---
# Claim: "CTR falls as position gets worse."
has_pos = df[df["avg_position"] > 0]  # drop the 'no position data' rows found above
pos_table = has_pos.groupby("position_tier").apply(
    lambda g: pd.Series({
        "n": len(g),
        "weighted_ctr_pct": g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100,
    })
).reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
verdict_line("Test 3: weighted CTR by position_tier (excl. no-position-data rows)", pos_table.round(3).to_string())


=== Test 1: staleness (freshness_tier) vs decline rate ===
                    n  decline_rate
freshness_tier                     
181+              174         0.471
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611


=== Test 2: volume (impression_tier) vs decline rate ===
                     n  decline_rate
impression_tier                     
low              11248         0.454
moderate         10469         0.615
good              7205         0.586
excellent         1078         0.462


=== Test 3: weighted CTR by position_tier (excl. no-position-data rows) ===
                     n  weighted_ctr_pct
position_tier                           
top_3           1116.0             0.489
page_1         11814.0             0.350
striking        7304.0             0.347
page_3_5        7242.0             0.155
deep            1319.0             0.041



/tmp/ipykernel_4236/1147410020.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pos_table = has_pos.groupby("position_tier").apply(


All tiers clear the n=50 floor. Not monotonic: the MOST stale tier (181+, n=174) has the
LOWEST decline rate (0.471); 91-180 (n=9,171) has the highest (0.611). Spearman corr of the
raw days_since_last_update against the label: 0.049 -- close to flat.
VERDICT: MIXED. Staleness alone does not cleanly predict decline in this snapshot.
All tiers clear the floor. Also not monotonic (low 0.454, moderate 0.615, good 0.586,
excellent 0.462 -- an inverted U). Spearman corr: 0.146 -- three times stronger than staleness's, and the
biggest single step (low -> moderate, n=11,248 and n=10,469) moves the right direction.
VERDICT: MIXED, leaning CONFIRMED. Real but not clean or linear.
All tiers clear the floor. This one IS monotonic and clean: 0.489% -> 0.350% -> 0.347% ->
0.155% -> 0.039% as position worsens from top_3 to deep.
VERDICT: CONFIRMED. Position genuinely drives CTR here -- the cleanest of the three.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [8]:
# The flag-linked test: does the CTR-fix flag's assumption hold?
#
# Claim behind the flag: "a page that's already well-positioned (top_3 / page_1 / striking) but
# earns LESS click-through than other pages at its own position tier is an under-optimized page --
# and under-optimized pages are more likely to be declining, not just under-earning."

well_positioned = has_pos[has_pos["position_tier"].isin(["top_3", "page_1", "striking"])].copy()

# Build each tier's own weighted-CTR benchmark, then compare every page against ITS tier's bar --
# never against the overall average, since tiers have very different baseline CTRs (test 3 above).
tier_benchmark = well_positioned.groupby("position_tier").apply(
    lambda g: g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100
)
print("Per-tier weighted CTR benchmark (%):")
print(tier_benchmark.round(3))
print()

well_positioned["tier_benchmark_ctr"] = well_positioned["position_tier"].map(tier_benchmark)
well_positioned["ctr_gap"] = well_positioned["ctr"] < well_positioned["tier_benchmark_ctr"]

gap_table = well_positioned.groupby("ctr_gap").agg(
    n=("content_id", "size"), decline_rate=("is_declining_label", "mean")
)
print(f"n = {len(well_positioned)} well-positioned pages (avg_position > 0, top_3/page_1/striking)")
print(gap_table.round(3))


Per-tier weighted CTR benchmark (%):
position_tier
page_1      0.350
striking    0.347
top_3       0.489
dtype: float64

n = 20234 well-positioned pages (avg_position > 0, top_3/page_1/striking)
             n  decline_rate
ctr_gap                     
False     5230         0.511
True     15004         0.604


/tmp/ipykernel_4236/3894451659.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  tier_benchmark = well_positioned.groupby("position_tier").apply(


Both groups clear the floor by a wide margin (5,230 and 15,004). Pages below their own
tier's CTR benchmark decline at 0.604 vs 0.511 for pages at/above benchmark -- a real,
meaningful gap, and it's the honest comparison (each page judged against its OWN tier's
bar, not a single blended average across very different baseline CTRs).
VERDICT: CONFIRMED. The CTR-fix flag's core assumption holds in this data: a well-ranked
page that's underperforming its tier's click-through really is more likely to be declining.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

What a content team should take from this:

1. Staleness and volume, used alone, are weak, noisy filters (MIXED verdicts) -- neither
   should gate a flag by itself. They're only useful combined with something else (as in
   the w04 baseline rule, which requires stale AND visible together).
2. The CTR-fix flag's assumption is the strongest of everything tested here (CONFIRMED,
   clean and monotonic): a well-positioned page underperforming its own tier's CTR is a
   real, worth-trusting signal -- and it should always be compared against a tier-specific
   benchmark, never a single site-wide CTR average, since baseline CTR varies a lot by
   position (0.49% at top_3 vs 0.04% deep).
3. Two data traps are worth fixing upstream before anyone trusts a position-based report:
   avg_position == 0 silently masquerading as a top_3 position, and any CTR figure that's
   a plain average of per-page rates instead of total-clicks/total-impressions.

## Self-check

Before you submit, confirm each line honestly:

- [Done ] Every section above is filled — markdown thinking AND the code that backs it
- [Done] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Done ] No client names, URLs, or private queries anywhere
- [Done ] My claims use careful words: observed, measured, directional, decision-support
- [Done ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.